# Northern Bangladesh Heart Disease — Machine Learning & Explainable AI

**Project:** AI-Driven Web-Based Heart Disease Prediction System Using Machine Learning: Design and Development of an Intelligent Clinical Decision Support Platform.

This Google Colab notebook implements the ML/XAI research component using the hospital-sourced Northern Bangladesh dataset. It compares Logistic Regression, Decision Tree, SVM and Random Forest using leakage-safe preprocessing, cross-validation, hyperparameter tuning, held-out testing and SHAP.

> **Research disclaimer:** This is an academic decision-support prototype. Predictions are not medical diagnoses and the model is not clinically validated.

## 1. Environment

In [1]:
import sys, os, json, warnings, subprocess
warnings.filterwarnings("ignore")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "shap", "joblib"], check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
OUTPUT_DIR = "heart_disease_ml_artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Ready. RANDOM_STATE =", RANDOM_STATE)

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load the Northern Bangladesh Excel Dataset

In [ ]:
from google.colab import files

EXPECTED = "Heart_diasease_dataset_from_Northern_Bangladesh.xlsx"
if os.path.exists(EXPECTED):
    DATA_PATH = EXPECTED
else:
    print("Upload the Northern Bangladesh Excel dataset.")
    uploaded = files.upload()
    excel_files = [x for x in uploaded if x.lower().endswith((".xlsx", ".xls"))]
    if not excel_files:
        raise FileNotFoundError("No Excel dataset uploaded.")
    DATA_PATH = excel_files[0]

xls = pd.ExcelFile(DATA_PATH)
SHEET = "Our Dataset" if "Our Dataset" in xls.sheet_names else xls.sheet_names[0]
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET)

print("Sheet:", SHEET)
print("Shape:", df_raw.shape)
display(df_raw.head())

## 3. Initial Data Quality Audit

In [ ]:
print("Shape:", df_raw.shape)
print("Exact duplicate rows:", df_raw.duplicated().sum())

display(df_raw.dtypes.to_frame("dtype"))

missing = df_raw.isna().sum().to_frame("missing_count")
missing["missing_percent"] = 100 * missing["missing_count"] / len(df_raw)
display(missing.sort_values("missing_percent", ascending=False))

print("Target distribution:")
display(df_raw["Heart Disease"].value_counts(dropna=False).rename("count").to_frame())

print("SL unique:", df_raw["SL"].nunique(), "/", len(df_raw))

## 4. Deterministic Cleaning and Column Normalization

In [ ]:
df = df_raw.copy()
df.columns = df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df = df.rename(columns={
    "Troponin-I": "Troponin_I",
    "Troponin- I assay type": "Troponin_Assay_Type",
    "Heart Disease": "Heart_Disease"
})

required = {
    "SL","Age","Sex","Height (cm)","Weight (kg)","BMI","Family H/O",
    "Hypertension","Diabetes","Total_Cholesterol(mg/dL)","BP(mmHg)",
    "H/O ChestPain","RBS(mmol/L)","HDL(mg/dL)","LDL(mg/dL)",
    "Triglycerides(mg/dL)","MaxHR","Himoglobin","Creatinine(mg/dL)",
    "Platelets","Sodium(mmol/L)","Potassium","Chloride","Troponin_I",
    "Troponin_Assay_Type","Heart_Disease","UNIT"
}
missing_required = required - set(df.columns)
assert not missing_required, f"Missing required columns: {missing_required}"

for col in ["Himoglobin", "Potassium", "Chloride"]:
    s = df[col].astype("string")
    s = s.str.replace("`", "", regex=False).str.replace(",", ".", regex=False)
    s = s.str.replace(r"(?<=\d)\s+\.(?=\d)", ".", regex=True)
    df[col] = pd.to_numeric(s, errors="coerce")

print(df.columns.tolist())
print("Numeric dtypes:")
display(df[["Himoglobin","Potassium","Chloride"]].dtypes.to_frame("dtype"))

## 5. Troponin-I Harmonization and Censoring

In [ ]:
s = df["Troponin_I"].astype("string").str.strip()
high = s.str.fullmatch(r">\s*25000").fillna(False)
low = s.str.fullmatch(r"<\s*2\.50").fillna(False)
ambiguous = s.str.fullmatch(r">\s*2\.5").fillna(False)

df["Troponin_Censored_High"] = high.astype(int)
df["Troponin_Censored_Low"] = low.astype(int)
df["Troponin_Censor_Ambiguous"] = ambiguous.astype(int)

t = pd.to_numeric(s.str.replace(r"^[<>]\s*", "", regex=True), errors="coerce")
t.loc[high] = 25000.0
t.loc[low] = 2.50
t.loc[ambiguous] = np.nan
df["Troponin_I"] = t

hs = df["Troponin_Assay_Type"].eq("High-Sensitivity Troponin-I (ng/L)")
df.loc[hs & df["Troponin_I"].notna(), "Troponin_I"] /= 1000.0

print("High censored:", int(high.sum()))
print("Low censored:", int(low.sum()))
print("Ambiguous >2.5:", int(ambiguous.sum()))
print("High-sensitivity records:", int(hs.sum()))

## 6. Pediatric Records

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
pediatric = df["Age"] < 18
print("Pediatric records:", int(pediatric.sum()))
if pediatric.sum():
    display(df.loc[pediatric, ["SL","Age","Heart_Disease"]])
    print("Pediatric target distribution:")
    display(df.loc[pediatric, "Heart_Disease"].value_counts().rename("count").to_frame())

df_model = df.loc[~pediatric].copy()
print("Modeling population:", df_model.shape)

## 7. Leakage Assessment and Feature Exclusion

In [ ]:
print("UNIT vs target:")
display(pd.crosstab(df_model["UNIT"], df_model["Heart_Disease"], margins=True))

print("Troponin assay type vs target:")
display(pd.crosstab(df_model["Troponin_Assay_Type"].fillna("<MISSING>"),
                    df_model["Heart_Disease"]))

# SL is an identifier. UNIT is outcome-associated admission context and is leakage.
# Assay type/missingness is also strongly outcome-associated in this dataset and is
# therefore excluded from predictive input; Troponin-I itself is retained.
DROP = ["SL", "UNIT", "Troponin_Assay_Type"]
df_model = df_model.drop(columns=DROP)

assert "Heart_Disease" in df_model
assert all(c not in df_model.columns for c in DROP)
print("Remaining predictors:", len(df_model.columns) - 1)

## 8. Missingness and Exploratory Analysis

In [ ]:
missing = df_model.isna().sum().to_frame("missing_count")
missing["missing_percent"] = 100 * missing["missing_count"] / len(df_model)
display(missing[missing.missing_count > 0].sort_values("missing_percent", ascending=False))

plt.figure(figsize=(9,5))
m = missing[missing.missing_count > 0].sort_values("missing_percent")
plt.barh(m.index, m.missing_percent)
plt.xlabel("Missing values (%)")
plt.title("Missingness by Predictor")
plt.tight_layout()
plt.show()

sns.countplot(data=df_model, x="Heart_Disease")
plt.title("Target Distribution")
plt.show()

num_cols = df_model.select_dtypes(include=np.number).columns.tolist()
num_predictors = [c for c in num_cols if c != "Heart_Disease"]

df_model[num_predictors].hist(figsize=(15,16), bins=25)
plt.tight_layout()
plt.show()

corr = df_model[num_predictors + ["Heart_Disease"]].corr(numeric_only=True)
plt.figure(figsize=(13,10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Numeric Correlation Matrix")
plt.tight_layout()
plt.show()

## 9. Correlation / Redundancy Screening

In [ ]:
abs_corr = corr.drop(index="Heart_Disease", errors="ignore").drop(columns="Heart_Disease", errors="ignore").abs()
upper = abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
pairs = (upper.stack().reset_index()
         .rename(columns={"level_0":"Feature_1","level_1":"Feature_2",0:"Absolute_Correlation"})
         .query("Absolute_Correlation >= 0.85")
         .sort_values("Absolute_Correlation", ascending=False))
if pairs.empty:
    print("No predictor pair reached |r| >= 0.85.")
else:
    display(pairs)
print("High correlation is treated as a screening flag; features are not automatically removed.")

## 10. Train/Test Split

In [ ]:
y = df_model["Heart_Disease"].astype(int)
X = df_model.drop(columns=["Heart_Disease"])

assert "Heart_Disease" not in X
assert "SL" not in X
assert "UNIT" not in X

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train class proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())
print("Test class proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 11. Leakage-Safe Preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

try:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(scale_numeric):
    num_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OHE)
    ])
    return ColumnTransformer([
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features)
    ], remainder="drop", verbose_feature_names_out=True)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 12. Four Required Models and Tuning Grids

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_specs = {
    "Logistic Regression": (
        Pipeline([
            ("preprocess", make_preprocessor(True)),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced",
                                         random_state=RANDOM_STATE))
        ]),
        {"model__C":[0.01,0.1,1,10], "model__solver":["liblinear"],
         "model__penalty":["l1","l2"]}
    ),
    "Decision Tree": (
        Pipeline([
            ("preprocess", make_preprocessor(False)),
            ("model", DecisionTreeClassifier(class_weight="balanced",
                                              random_state=RANDOM_STATE))
        ]),
        {"model__max_depth":[3,5,8,None],
         "model__min_samples_split":[2,10],
         "model__min_samples_leaf":[1,5,10]}
    ),
    "SVM": (
        Pipeline([
            ("preprocess", make_preprocessor(True)),
            ("model", SVC(probability=True, class_weight="balanced",
                          random_state=RANDOM_STATE))
        ]),
        {"model__C":[0.1,1,10], "model__kernel":["linear","rbf"],
         "model__gamma":["scale","auto"]}
    ),
    "Random Forest": (
        Pipeline([
            ("preprocess", make_preprocessor(False)),
            ("model", RandomForestClassifier(class_weight="balanced",
                                              n_estimators=300,
                                              n_jobs=-1,
                                              random_state=RANDOM_STATE))
        ]),
        {"model__n_estimators":[200,400],
         "model__max_depth":[None,5,10],
         "model__min_samples_leaf":[1,5],
         "model__max_features":["sqrt","log2"]}
    )
}
print(list(model_specs))

## 13. Hyperparameter Tuning — ROC-AUC

In [ ]:
searches = {}
rows = []

for name, (pipe, grid) in model_specs.items():
    print("Tuning:", name)
    search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv,
                          n_jobs=-1, refit=True, return_train_score=False)
    search.fit(X_train, y_train)
    searches[name] = search
    rows.append({"Model":name, "Best_CV_ROC_AUC":search.best_score_,
                 "Best_Params":search.best_params_})
    print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

tuning_summary = pd.DataFrame(rows).sort_values("Best_CV_ROC_AUC", ascending=False)
display(tuning_summary)

## 14. Cross-Validation Stability

In [ ]:
scoring = {
    "accuracy":"accuracy", "precision":"precision", "recall":"recall",
    "f1":"f1", "roc_auc":"roc_auc"
}
cv_rows = []

for name, search in searches.items():
    r = cross_validate(search.best_estimator_, X_train, y_train, cv=cv,
                       scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Model":name,
        "CV Accuracy Mean":r["test_accuracy"].mean(), "CV Accuracy SD":r["test_accuracy"].std(),
        "CV Precision Mean":r["test_precision"].mean(), "CV Precision SD":r["test_precision"].std(),
        "CV Recall Mean":r["test_recall"].mean(), "CV Recall SD":r["test_recall"].std(),
        "CV F1 Mean":r["test_f1"].mean(), "CV F1 SD":r["test_f1"].std(),
        "CV ROC-AUC Mean":r["test_roc_auc"].mean(), "CV ROC-AUC SD":r["test_roc_auc"].std()
    })

cv_summary = pd.DataFrame(cv_rows).sort_values(
    ["CV ROC-AUC Mean","CV Recall Mean","CV F1 Mean"], ascending=False
)
display(cv_summary.round(4))

## 15. Held-Out Test Evaluation

In [ ]:
test_rows = []
predictions, probabilities = {}, {}

for name, search in searches.items():
    est = search.best_estimator_
    pred = est.predict(X_test)
    prob = est.predict_proba(X_test)[:,1]
    predictions[name], probabilities[name] = pred, prob
    test_rows.append({
        "Model":name,
        "Test Accuracy":accuracy_score(y_test,pred),
        "Test Precision":precision_score(y_test,pred,zero_division=0),
        "Test Recall":recall_score(y_test,pred,zero_division=0),
        "Test F1":f1_score(y_test,pred,zero_division=0),
        "Test ROC-AUC":roc_auc_score(y_test,prob)
    })

test_summary = pd.DataFrame(test_rows)
comparison = cv_summary.merge(test_summary,on="Model").sort_values(
    ["CV ROC-AUC Mean","CV Recall Mean","CV F1 Mean"], ascending=False
)
display(comparison.round(4))

## 16. Confusion Matrices, Classification Reports and ROC Curves

In [ ]:
for name in searches:
    print("="*70, name, "="*70)
    print(classification_report(y_test, predictions[name],
                                target_names=["Negative","Positive"],
                                zero_division=0))
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, predictions[name]),
        display_labels=["Negative","Positive"]
    ).plot()
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

plt.figure(figsize=(9,7))
for name, prob in probabilities.items():
    fpr,tpr,_ = roc_curve(y_test,prob)
    auc = roc_auc_score(y_test,prob)
    plt.plot(fpr,tpr,label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"--",label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Held-Out Test ROC Curves")
plt.legend()
plt.show()

## 17. Final Model Selection

In [ ]:
selection = comparison.copy()
selection["Test_CV_ROC_AUC_Abs_Gap"] = (
    selection["Test ROC-AUC"] - selection["CV ROC-AUC Mean"]
).abs()

selection = selection.sort_values(
    ["CV ROC-AUC Mean","CV Recall Mean","CV F1 Mean","Test ROC-AUC"],
    ascending=False
).reset_index(drop=True)

FINAL_MODEL_NAME = selection.loc[0,"Model"]
FINAL_PIPELINE = searches[FINAL_MODEL_NAME].best_estimator_

display(selection[[
    "Model","CV ROC-AUC Mean","CV ROC-AUC SD","CV Recall Mean","CV F1 Mean",
    "Test ROC-AUC","Test Recall","Test F1","Test_CV_ROC_AUC_Abs_Gap"
]].round(4))

print("Selected model:", FINAL_MODEL_NAME)
print("Selection priority: CV ROC-AUC → CV Recall → CV F1 → test ROC-AUC.")
print("This is a research-model selection decision, not a claim of clinical superiority.")

## 18. SHAP Explainability

In [ ]:
import shap

pre = FINAL_PIPELINE.named_steps["preprocess"]
clf = FINAL_PIPELINE.named_steps["model"]

feature_names = np.array(pre.get_feature_names_out())
X_test_t = pre.transform(X_test)

rng = np.random.default_rng(RANDOM_STATE)
n_bg = min(50, len(X_train))
n_exp = min(75, len(X_test))
bg_idx = rng.choice(len(X_train), size=n_bg, replace=False)
X_bg = pre.transform(X_train.iloc[bg_idx])
X_exp = X_test_t[:n_exp]

if isinstance(clf, (DecisionTreeClassifier, RandomForestClassifier)):
    explainer = shap.TreeExplainer(clf)
    raw = explainer.shap_values(X_exp)
    if isinstance(raw, list):
        shap_values = np.asarray(raw[1])
    else:
        arr = np.asarray(raw)
        shap_values = arr[:,:,1] if arr.ndim == 3 else arr
else:
    predict_fn = lambda z: clf.predict_proba(z)[:,1]
    explainer = shap.Explainer(predict_fn, X_bg, feature_names=feature_names)
    result = explainer(X_exp, max_evals=max(2*X_exp.shape[1]+1,100))
    shap_values = np.asarray(result.values)

if shap_values.ndim == 3:
    shap_values = shap_values[:,:, -1]

print("Selected model:", FINAL_MODEL_NAME)
print("SHAP matrix:", shap_values.shape)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_exp, feature_names=feature_names, show=False)
plt.title(f"SHAP Summary — {FINAL_MODEL_NAME}")
plt.tight_layout()
plt.show()

global_importance = pd.DataFrame({
    "Feature":feature_names,
    "Mean_Absolute_SHAP":np.abs(shap_values).mean(axis=0)
}).sort_values("Mean_Absolute_SHAP",ascending=False)

display(global_importance.head(15))

In [ ]:
try:
    base = 0.0
    if hasattr(explainer, "expected_value"):
        ev = np.asarray(explainer.expected_value).reshape(-1)
        base = float(ev[-1])
    shap.waterfall_plot(
        shap.Explanation(values=shap_values[0], base_values=base,
                         data=X_exp[0], feature_names=feature_names),
        max_display=15, show=False
    )
    plt.title("Local SHAP Explanation — First Held-Out Test Record")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Local waterfall could not be rendered:", e)

## 19. Example Prediction

In [ ]:
example = X_test.iloc[[0]]
pred = int(FINAL_PIPELINE.predict(example)[0])
prob = float(FINAL_PIPELINE.predict_proba(example)[0,1])

display(example.T.rename(columns={example.index[0]:"value"}))
print("Actual held-out label:", int(y_test.iloc[0]))
print("Predicted class:", pred)
print(f"Predicted positive-class probability: {prob:.4f}")
print("This is a research-model prediction, not a medical diagnosis.")

## 20. Save Complete Inference Pipeline and Metadata

In [ ]:
MODEL_PATH = os.path.join(OUTPUT_DIR,"heart_disease_final_pipeline.joblib")
METADATA_PATH = os.path.join(OUTPUT_DIR,"model_metadata.json")
CLEAN_PATH = os.path.join(OUTPUT_DIR,"northern_bangladesh_cleaned_modeling_population.csv")
RESULTS_PATH = os.path.join(OUTPUT_DIR,"model_comparison_results.csv")

joblib.dump(FINAL_PIPELINE, MODEL_PATH)
df_model.to_csv(CLEAN_PATH,index=False)
comparison.to_csv(RESULTS_PATH,index=False)

metadata = {
    "project":"AI-Driven Web-Based Heart Disease Prediction System",
    "dataset":"Northern Bangladesh hospital-sourced dataset",
    "target":"Heart_Disease",
    "random_state":RANDOM_STATE,
    "test_size":0.20,
    "cv_folds":5,
    "selected_model":FINAL_MODEL_NAME,
    "excluded_columns":["SL","UNIT","Troponin_Assay_Type"],
    "pediatric_rule":"Age < 18 excluded from modeling population",
    "troponin_model_unit":"ng/mL",
    "models_compared":list(searches.keys()),
    "raw_input_features":X.columns.tolist(),
    "transformed_feature_count":len(feature_names),
    "transformed_features":feature_names.tolist(),
    "best_parameters":searches[FINAL_MODEL_NAME].best_params_,
    "artifact":MODEL_PATH
}
with open(METADATA_PATH,"w",encoding="utf-8") as f:
    json.dump(metadata,f,indent=2)

print("Saved artifacts:")
for p in [MODEL_PATH,METADATA_PATH,CLEAN_PATH,RESULTS_PATH]:
    print(p)

## 21. Django Integration Schema

In [ ]:
schema = pd.DataFrame({
    "feature":X.columns,
    "dtype":[str(X[c].dtype) for c in X.columns]
})
display(schema)

print("The Django backend should load the serialized complete pipeline:")
print(MODEL_PATH)
print("The frontend should contain no independent prediction logic.")

## 22. Optional Downloads

In [ ]:
from google.colab import files as colab_files

# Uncomment the files you want to download:
# colab_files.download(MODEL_PATH)
# colab_files.download(METADATA_PATH)
# colab_files.download(CLEAN_PATH)
# colab_files.download(RESULTS_PATH)

print("Notebook complete.")
print("Selected model:", FINAL_MODEL_NAME)
print("Research reminder: model performance is dataset-specific and does not establish clinical validity.")